### Simple GenAI App Using LangChain And OpenAI Model

In [ ]:
# LangChain imports
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Community integrations
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS

# OpenAI integrations
from langchain_openai import ChatOpenAI
from langchain_openai.embeddings import OpenAIEmbeddings

# Environment variables
from dotenv import load_dotenv

load_dotenv()

In [ ]:
# Scraping the target website for the document
loader = WebBaseLoader("https://docs.langchain.com/langsmith/observability-llm-tutorial")
docs = loader.load()
docs

In [ ]:
# Splitting the document into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = text_splitter.split_documents(docs)
documents

In [ ]:
# Converting the chunks into vectors and storing it in the vector store
embeddings = OpenAIEmbeddings()
vector_store = FAISS.from_documents(documents, embeddings)

In [ ]:
# Querying from the vector store
query = "Once your app is working well in prototyping, you release it to a small group of real users."
result = vector_store.similarity_search_with_score(query)
result[0].page_content

In [ ]:
# Converting the vector store into retriever
retriever = vector_store.as_retriever()

In [ ]:
# Initializing the openai llm model
llm = ChatOpenAI(model='gpt-4o')
print(llm)

In [ ]:
# Creating the custom prompt template
prompt = ChatPromptTemplate.from_template(
    """
    Answer the following questions based on the provided context:
    <context>
    {context}
    </context>
    Question: {input}
    """
)

In [ ]:
# Creating the document chain
document_chain = create_stuff_documents_chain(llm, prompt)
document_chain

In [ ]:
# Creating the retrieval chain
retrieval_chain = create_retrieval_chain(retriever, document_chain)

In [ ]:
# Invoking the retrieval chain
response = retrieval_chain.invoke(
    {
        "input": "Once your app is working well in prototyping, you release it to a small group of real users."
    }
)

response["answer"]

##### Langchain Document and Retrieval Chain have Depreciated and Replaced With Langchain Expression Language (LCEL)